# Notebook 5: Hybrid LSTM-SNP (Fuzzy Feature Aug + Fuzzy Gates)

**Dataset**: Dow Jones Industrial Index

## Description
This notebook implements the **hybrid model** that combines:
1. **Fuzzy Feature Augmentation** (from Notebook 2): Input is augmented with fuzzy-derived features
2. **Fuzzy Gate Replacement** (from Notebook 3): Gates use fuzzy inference instead of sigmoid

This represents the most complex integration of fuzzy logic with LSTM-SNP. Gradient clipping 
(norm=1.0) and careful weight initialization are applied for training stability.

The interaction between input-level and gate-level fuzzy reasoning is documented and analyzed.

## Theory: Hybrid Fuzzy LSTM-SNP

This model combines two fuzzy integration strategies:

### 1. Input-Level: Fuzzy Feature Augmentation
Same as Notebook 2:
- Fuzzy inference on $x(t)$ and $x(t-1)$
- Augmented input: $x'(t) = [x(t), y_{fuzzy}]$
- Input dimension becomes 2

### 2. Gate-Level: Fuzzy Gate Replacement
Same as Notebook 3:
- Gates r, c, o computed via fuzzy inference
- Fixed Gaussian MFs, trainable consequent parameters
- Generation gate a retains tanh

### Interaction Between Levels
- The fuzzy-augmented input provides richer information to the cell
- The fuzzy gates process this enriched input with adaptive, rule-based gating
- This creates a two-level fuzzy reasoning pipeline

### Stability Measures
- Gradient clipping (norm ≤ 1.0)
- Careful weight initialization
- Fuzzy weights are clipped to valid ranges

## Model Architecture & Implementation

In [15]:
# ============================================================
# ALL IMPORTS
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from math import sqrt
from scipy import stats
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

PyTorch version: 2.11.0+cu128
NumPy version: 2.5.1
Using device: cuda


### Fuzzy Inference System (NumPy — for preprocessing)

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# Fuzzy Inference System (NumPy — for preprocessing)
#
# Fixed Gaussian membership functions:
#   μ_low(x)  = exp(-(x - (-1))² / (2·0.5²))
#   μ_high(x) = exp(-(x - (+1))² / (2·0.5²))
#
# 4 Takagi-Sugeno rules with fixed consequent parameters:
#   IF x(t) is low  AND x(t-1) is low  → y₁ = 0.5·x(t) + 0.5·x(t-1)
#   IF x(t) is low  AND x(t-1) is high → y₂ = 0.7·x(t) + 0.3·x(t-1) - 0.1
#   IF x(t) is high AND x(t-1) is low  → y₃ = 0.3·x(t) + 0.7·x(t-1) + 0.1
#   IF x(t) is high AND x(t-1) is high → y₄ = 0.5·x(t) + 0.5·x(t-1)
#
# Output: y = Σ(wᵢ·yᵢ) / Σ(wᵢ)
# ============================================================

def gaussian_mf(x, center, sigma=0.5):
    """Fixed Gaussian membership function."""
    return np.exp(-(x - center)**2 / (2 * sigma**2))

def fuzzy_inference_numpy(x_t, x_tm1):
    """
    Compute fuzzy feature from x(t) and x(t-1).
    Uses fixed membership functions and fixed consequent parameters.
    """
    # Membership degrees
    mu_low_xt = gaussian_mf(x_t, center=-1.0)
    mu_high_xt = gaussian_mf(x_t, center=1.0)
    mu_low_xtm1 = gaussian_mf(x_tm1, center=-1.0)
    mu_high_xtm1 = gaussian_mf(x_tm1, center=1.0)

    # Rule firing strengths (product)
    w1 = mu_low_xt * mu_low_xtm1      # low-low
    w2 = mu_low_xt * mu_high_xtm1     # low-high
    w3 = mu_high_xt * mu_low_xtm1     # high-low
    w4 = mu_high_xt * mu_high_xtm1    # high-high

    # Consequent outputs (fixed linear functions)
    y1 = 0.5 * x_t + 0.5 * x_tm1
    y2 = 0.7 * x_t + 0.3 * x_tm1 - 0.1
    y3 = 0.3 * x_t + 0.7 * x_tm1 + 0.1
    y4 = 0.5 * x_t + 0.5 * x_tm1

    # Weighted average defuzzification
    numerator = w1 * y1 + w2 * y2 + w3 * y3 + w4 * y4
    denominator = w1 + w2 + w3 + w4 + 1e-8

    return numerator / denominator

### Fuzzy Gate LSTM-SNP Cell

In [17]:
# ============================================================
# Fuzzy Gate LSTM-SNP Cell (PyTorch Implementation)
# Gates r, c, o use fuzzy inference instead of hard sigmoid.
# Generation gate 'a' keeps tanh (unchanged).
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

class FuzzyGateType5(nn.Module):
    def __init__(self, hidden_size, sigma=0.5):
        super().__init__()
        self.hidden_size = hidden_size
        self.sigma = sigma
        self.mu_low = -1.0
        self.mu_high = 1.0
        
        # Rule 0 (Low)
        self.a0 = nn.Parameter(torch.empty(hidden_size).uniform_(-0.1, 0.1))
        self.b0 = nn.Parameter(torch.empty(hidden_size).uniform_(-0.1, 0.1))
        self.c0 = nn.Parameter(torch.zeros(hidden_size))
        
        # Rule 1 (High)
        self.a1 = nn.Parameter(torch.empty(hidden_size).uniform_(-0.1, 0.1))
        self.b1 = nn.Parameter(torch.empty(hidden_size).uniform_(-0.1, 0.1))
        self.c1 = nn.Parameter(torch.zeros(hidden_size))

    def _gaussian_mf(self, x, center):
        return torch.exp(-(x - center)**2 / (2.0 * self.sigma**2))

    def forward(self, z_gate, u_mean):
        w_low = self._gaussian_mf(z_gate, self.mu_low)
        w_high = self._gaussian_mf(z_gate, self.mu_high)
        
        y0 = self.a0 * z_gate + self.b0 * u_mean + self.c0
        y1 = self.a1 * z_gate + self.b1 * u_mean + self.c1
        
        numerator = w_low * y0 + w_high * y1
        denominator = w_low + w_high + 1e-8
        output = numerator / denominator
        return torch.clamp(output, 0.0, 1.0)


class FuzzyLSTMSNPCell(nn.Module):
    """
    LSTM-SNP Cell with fuzzy gate replacement.
    Gates r, c, o are computed via fuzzy inference.
    Gate a keeps tanh activation.
    """
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        self.W = nn.Linear(input_size, 4 * hidden_size, bias=True)
        self.U = nn.Linear(hidden_size, 4 * hidden_size, bias=True)
        
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.orthogonal_(self.U.weight)
        nn.init.zeros_(self.W.bias)
        nn.init.zeros_(self.U.bias)
        
        self.fuzzy_r = FuzzyGateType5(hidden_size)
        self.fuzzy_c = FuzzyGateType5(hidden_size)
        self.fuzzy_o = FuzzyGateType5(hidden_size)

    def forward(self, x, u_prev):
        z = self.W(x) + self.U(u_prev)
        
        z0 = z[:, :self.hidden_size]
        z1 = z[:, self.hidden_size:2*self.hidden_size]
        z2 = z[:, 2*self.hidden_size:3*self.hidden_size]
        z3 = z[:, 3*self.hidden_size:]
        
        # Mean of previous state for fuzzy rule input
        u_mean = u_prev.mean(dim=-1, keepdim=True).expand_as(u_prev)
        
        r = self.fuzzy_r(z0, u_mean)
        c = self.fuzzy_c(z1, u_mean)
        o = self.fuzzy_o(z2, u_mean)
        a = torch.tanh(z3)
        
        u = r * u_prev - c * a
        h = o * a
        
        return h, u


### Build Model

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# Model Construction (PyTorch)
# ============================================================

class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        
        # Select cell
        if 5 in [1, 2, 4]:
            self.cell = LSTMSNPCell(input_size, hidden_size)
        else:
            self.cell = FuzzyLSTMSNPCell(input_size, hidden_size)
            
        # Select output layer
        if 5 == 4:
            self.out = FuzzyOutputLayer(hidden_size)
        else:
            self.out = nn.Linear(hidden_size, 1)
            
        self.u = None
        
    def reset_states(self, batch_size, device):
        self.u = torch.zeros(batch_size, self.hidden_size, device=device)

    def detach_states(self):
        # Truncate BPTT: keep the state's value, drop its graph history
        if self.u is not None:
            self.u = self.u.detach()
        
    def forward(self, x):
        # x is (batch, 1, input_size)
        if self.u is None or self.u.device != x.device:
            self.reset_states(x.size(0), x.device)
            
        h, self.u = self.cell(x[:, 0, :], self.u)
        return self.out(h)
        
def build_model(input_dim, units):
    return RNNModel(input_dim, units)

In [19]:
# Quick model check
model = build_model(input_dim=2, units=8)
print(model)

RNNModel(
  (cell): FuzzyLSTMSNPCell(
    (W): Linear(in_features=2, out_features=32, bias=True)
    (U): Linear(in_features=8, out_features=32, bias=True)
    (fuzzy_r): FuzzyGateType5()
    (fuzzy_c): FuzzyGateType5()
    (fuzzy_o): FuzzyGateType5()
  )
  (out): Linear(in_features=8, out_features=1, bias=True)
)


## Data Pipeline — Dow Jones Industrial Index

In [20]:
# ============================================================
# 1. Load Time Series Data
# ============================================================
series = pd.read_csv(
    r'C:\Users\paulp\OneDrive\Desktop\fuzzy_LSTM\dataset\monthly-closings-of-the-dowjones.csv',
    header=0,
    parse_dates=[0],
    index_col=0
)
raw_values = series.values.flatten()
print(f"Data shape: {raw_values.shape}")
print(f"First 5 values: {raw_values[:5]}")

Data shape: (291,)
First 5 values: [3645.  3626.  3634.  3620.5 3607. ]


In [21]:
# ============================================================
# 2. First-Order Differencing
# ============================================================

def difference(dataset, interval=1):
    diff = []
    for i in range(interval, len(dataset)):
        value = dataset[i] - dataset[i - interval]
        diff.append(value)
    return np.array(diff)

In [22]:
# ============================================================
# 3. Convert to Supervised Learning Format (lag=1)
# ============================================================

def timeseries_to_supervised(data, lag=1):
    df = pd.DataFrame(data)
    columns = [df.shift(i) for i in range(1, lag+1)]
    columns.append(df)
    df = pd.concat(columns, axis=1)
    df.fillna(0, inplace=True)
    return df.values

In [23]:
# ============================================================
# 3b. Apply differencing + supervised-format conversion
# (evaluate_on_test() adds raw_values[t] back to predictions,
# which only makes sense if the model is trained on first
# differences -- so that step has to happen here.)
# ============================================================
diff_values = difference(raw_values, interval=1)
supervised = timeseries_to_supervised(diff_values, lag=5)
print(f"Differenced series: {diff_values.shape}, supervised: {supervised.shape}")

Differenced series: (290,), supervised: (290, 6)


In [24]:
# ============================================================
# 4. Train-Validation-Test Split (Last 60 points as Test)
# ============================================================
n = len(supervised)
test_size = 60
train_val = supervised[:n - test_size]
test = supervised[n - test_size:]

val_size = int(len(train_val) * 0.1)
train = train_val[:-val_size]
val = train_val[-val_size:]

print(f"Train: {train.shape}, Val: {val.shape}, Test: {test.shape}")

# ============================================================
# 5. Feature Scaling
# ============================================================
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(train)  # fit only on train to avoid leakage
train_scaled = scaler.transform(train)
val_scaled = scaler.transform(val)
test_scaled = scaler.transform(test)

Train: (207, 6), Val: (23, 6), Test: (60, 6)


In [25]:
# ============================================================
# 6. Reshape for RNN Input
# ============================================================

X_train, y_train = train_scaled[:, 0:-1], train_scaled[:, -1]
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))

X_test, y_test = test_scaled[:, 0:-1], test_scaled[:, -1]
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (207, 1, 5), y_train shape: (207,)


In [26]:
#Gaussian noise injection (Eq. 29–30)
ref_std = X_train.std()
print(f"Reference std(x) used for noise scaling (train inputs): {ref_std:.6f}")

NOISE_LEVELS = [0.005, 0.05, 0.10, 0.15]  # 0.5%, 5%, 10%, 15%

def add_gaussian_noise(X, noise_level, ref_std, seed=None):
    """
    x_noisy(t) = x(t) + eps(t),  eps(t) ~ N(0, sigma_eps^2),  sigma_eps = noise_level * std(x)
    Applied to input features X only — never to targets/labels.
    """
    rng = np.random.default_rng(seed)
    sigma_eps = noise_level * ref_std
    eps = rng.normal(loc=0.0, scale=sigma_eps, size=X.shape)
    return X + eps

Reference std(x) used for noise scaling (train inputs): 0.291104


In [27]:
#Evaluation function

def evaluate_on_test(model, X_test_eval, train, train_scaled, raw_values, scaler, device):
    """
    Warms up the model's recurrent state on clean training data, then runs
    single-step-ahead prediction on X_test_eval (clean or Gaussian-noise-corrupted
    input features). Test targets are always the clean, ground-truth values.
    """
    model.eval()
    model.reset_states(1, device)

    with torch.no_grad():
        for i in range(len(train_scaled)):
            X_raw = train_scaled[i, 0:-1]
            X_input = torch.tensor(X_raw, dtype=torch.float32).view(1, 1, len(X_raw)).to(device)
            model(X_input)

        predictions = []
        for i in range(len(X_test_eval)):
            X = X_test_eval[i]
            X_input = torch.tensor(X, dtype=torch.float32).view(1, 1, len(X)).to(device)
            yhat = model(X_input).item()

            new_row = [x for x in X] + [yhat]
            array = np.array(new_row).reshape(1, len(new_row))
            inverted = scaler.inverse_transform(array)[0, -1]
            inverted = inverted + raw_values[len(train) + i]
            predictions.append(inverted)

    actual = raw_values[-len(X_test_eval):]
    mse = mean_squared_error(actual, predictions)
    rmse = sqrt(mse)
    meanV = np.mean(actual)
    dominator = np.linalg.norm(np.array(actual) - meanV, 2)   # FIX: was predictions - meanV
    nmse = mse / np.power(dominator, 2)
    return predictions, rmse, mse, nmse

## Training Loop

In [28]:
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CHECKPOINT_DIR = "checkpoints_dow_jones"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def _ckpt_path(run):
    return os.path.join(CHECKPOINT_DIR, f"run_{run}.pt")

def _summary_path():
    return os.path.join(CHECKPOINT_DIR, "summary.pt")

all_rmse = []
all_mse = []
all_nmse = []
all_predictions = []
all_losses = []
all_models = []  # keep every trained model for later noise-robustness eval

print(f"\n--- [PyTorch] RUNNING ON {device} ---\n")
print(f"Checkpoints will be read/written under: {os.path.abspath(CHECKPOINT_DIR)}\n")

N_RUNS = 60
val_X = val_scaled[:, 0:-1]
PATIENCE = 10
INPUT_DIM = X_train.shape[-1]  # derived from data (== lag), not hardcoded

summary_path = _summary_path()
if os.path.exists(summary_path):
    print("Found completed checkpoint summary -> loading, skipping training.")
    saved = torch.load(summary_path, map_location=device, weights_only=False)

    all_models = []
    for sd in saved['model_state_dicts']:
        m = build_model(input_dim=INPUT_DIM, units=8).to(device)
        m.load_state_dict(sd)
        m.eval()
        all_models.append(m)

    all_rmse = saved['rmse']
    all_mse = saved['mse']
    all_nmse = saved['nmse']
    all_predictions = saved['predictions']
    all_losses = saved['losses']
    print(f"Loaded {len(all_models)} models from checkpoint.")

else:
    for run in range(N_RUNS):
        run_ckpt_path = _ckpt_path(run)
        if os.path.exists(run_ckpt_path):
            print(f'\n===== RUN {run + 1}/{N_RUNS} (resumed from checkpoint) =====')
            ckpt = torch.load(run_ckpt_path, map_location=device, weights_only=False)

            model = build_model(input_dim=INPUT_DIM, units=8).to(device)
            model.load_state_dict(ckpt['model_state_dict'])
            model.eval()

            all_losses.append(ckpt['run_losses'])
            all_models.append(model)
            all_rmse.append(ckpt['rmse'])
            all_mse.append(ckpt['mse'])
            all_nmse.append(ckpt['nmse'])
            all_predictions.append(ckpt['predictions'])
            print(f"Loaded run {run + 1} -- RMSE: {ckpt['rmse']:.6f}, MSE: {ckpt['mse']:.6f}, NMSE: {ckpt['nmse']:.10f} "
                  f"(best val RMSE: {ckpt['best_val_rmse']:.6f})")
            continue

        print(f'\n===== RUN {run + 1}/{N_RUNS} =====')

        np.random.seed(run)
        torch.manual_seed(run)
        model = build_model(input_dim=INPUT_DIM, units=8).to(device)
        # No LSTM-SNP-style gate-bias init here -- nn.LSTM's own default
        # initialization is used; PureLSTM has no .cell/.hidden_size to slice into.

        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.MSELoss()
        run_losses = []

        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
        n_samples = X_train_t.size(0)

        best_val_rmse = float('inf')
        best_state = None
        patience_ctr = 0

        for epoch in range(100):
            model.train()
            model.reset_states(1, device)
            epoch_loss = 0.0

            for i in range(n_samples):
                x_i = X_train_t[i:i + 1]
                y_i = y_train_t[i:i + 1]

                optimizer.zero_grad()
                pred = model(x_i)
                loss = criterion(pred.squeeze(-1), y_i)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                model.detach_states()  # truncate BPTT between steps
                epoch_loss += loss.item()

            avg_loss = epoch_loss / n_samples
            run_losses.append(avg_loss)

            _, val_rmse, _, _ = evaluate_on_test(model, val_X, train, train_scaled, raw_values, scaler, device)
            print(f"Epoch {epoch + 1}/100 — train loss: {avg_loss:.6f}  val RMSE: {val_rmse:.6f}")

            if val_rmse < best_val_rmse:
                best_val_rmse = val_rmse
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    print(f"Early stopping at epoch {epoch + 1} (best val RMSE: {best_val_rmse:.6f})")
                    break

        model.load_state_dict(best_state)
        model.reset_states(1, device)

        all_losses.append(run_losses)
        all_models.append(model)
        print(f'Training complete for run {run + 1} (best val RMSE: {best_val_rmse:.6f})')

        predictions, rmse, mse, nmse = evaluate_on_test(
            model, X_test, train, train_scaled, raw_values, scaler, device
        )

        all_rmse.append(rmse)
        all_mse.append(mse)
        all_nmse.append(nmse)
        all_predictions.append(predictions)

        print(f'Run {run + 1} — RMSE: {rmse:.6f}, MSE: {mse:.6f}, NMSE: {nmse:.10f}')

        torch.save({
            'model_state_dict': model.state_dict(),
            'run_losses': run_losses,
            'rmse': rmse,
            'mse': mse,
            'nmse': nmse,
            'predictions': predictions,
            'best_val_rmse': best_val_rmse,
        }, run_ckpt_path)

    torch.save({
        'rmse': all_rmse,
        'mse': all_mse,
        'nmse': all_nmse,
        'predictions': all_predictions,
        'losses': all_losses,
        'model_state_dicts': [m.state_dict() for m in all_models],
    }, summary_path)


--- [PyTorch] RUNNING ON cuda ---

Checkpoints will be read/written under: c:\Users\paulp\OneDrive\Desktop\fuzzy_LSTM\Type_5\checkpoints_dow_jones


===== RUN 1/60 =====
Epoch 1/100 — train loss: 0.131624  val RMSE: 162.564974
Epoch 2/100 — train loss: 0.090891  val RMSE: 159.756353
Epoch 3/100 — train loss: 0.089936  val RMSE: 159.543359
Epoch 4/100 — train loss: 0.089529  val RMSE: 159.412643
Epoch 5/100 — train loss: 0.089202  val RMSE: 159.313041
Epoch 6/100 — train loss: 0.088928  val RMSE: 159.236571
Epoch 7/100 — train loss: 0.088693  val RMSE: 159.174745
Epoch 8/100 — train loss: 0.088492  val RMSE: 159.132981
Epoch 9/100 — train loss: 0.088318  val RMSE: 159.084662
Epoch 10/100 — train loss: 0.088157  val RMSE: 159.081554
Epoch 11/100 — train loss: 0.088056  val RMSE: 159.010545
Epoch 12/100 — train loss: 0.087940  val RMSE: 158.964429
Epoch 13/100 — train loss: 0.087827  val RMSE: 158.935270
Epoch 14/100 — train loss: 0.087745  val RMSE: 158.894922
Epoch 15/100 — train loss:

## Results

In [29]:
#Noise-robustness sweep (averaged over multiple noise draws, with explicit clean baseline)

N_NOISE_DRAWS = 10  # average over multiple ε(t) realizations per model, per level

# Re-evaluate the clean (0% noise) baseline through evaluate_on_test, so it
# uses the exact same evaluation path as the noisy conditions below.
clean_rmse, clean_mse, clean_nmse = [], [], []
for model in all_models:
    _, rmse, mse, nmse = evaluate_on_test(model, X_test, train, train_scaled, raw_values, scaler, device)
    clean_rmse.append(rmse)
    clean_mse.append(mse)
    clean_nmse.append(nmse)

print(f"[{'no noise':>10}]  "
      f"RMSE: {np.mean(clean_rmse):.6f} ± {np.std(clean_rmse):.6f}  |  "
      f"MSE: {np.mean(clean_mse):.6f} ± {np.std(clean_mse):.6f}  |  "
      f"NMSE: {np.mean(clean_nmse):.10f} ± {np.std(clean_nmse):.10f}")

noise_robustness = {}  # noise_level -> dict of per-model RMSE/MSE/NMSE (averaged over draws)

for noise_level in NOISE_LEVELS:
    level_rmse, level_mse, level_nmse = [], [], []

    for run, model in enumerate(all_models):
        draw_rmse, draw_mse, draw_nmse = [], [], []
        for draw in range(N_NOISE_DRAWS):
            seed = hash((noise_level, run, draw)) % (2 ** 32)
            X_test_noisy = add_gaussian_noise(X_test, noise_level, ref_std, seed=seed)
            _, rmse, mse, nmse = evaluate_on_test(
                model, X_test_noisy, train, train_scaled, raw_values, scaler, device
            )
            draw_rmse.append(rmse)
            draw_mse.append(mse)
            draw_nmse.append(nmse)

        level_rmse.append(np.mean(draw_rmse))  # this model's average over noise draws
        level_mse.append(np.mean(draw_mse))
        level_nmse.append(np.mean(draw_nmse))

    noise_robustness[noise_level] = {'rmse': level_rmse, 'mse': level_mse, 'nmse': level_nmse}
    print(f"[{noise_level * 100:5.1f}% noise]  "
          f"RMSE: {np.mean(level_rmse):.6f} ± {np.std(level_rmse):.6f}  |  "
          f"MSE: {np.mean(level_mse):.6f} ± {np.std(level_mse):.6f}  |  "
          f"NMSE: {np.mean(level_nmse):.10f} ± {np.std(level_nmse):.10f}")

[  no noise]  RMSE: 100.124577 ± 1.192504  |  MSE: 10026.352999 ± 239.579135  |  NMSE: 0.0464272756 ± 0.0011093771
[  0.5% noise]  RMSE: 100.124597 ± 1.192390  |  MSE: 10026.356798 ± 239.555005  |  NMSE: 0.0464272932 ± 0.0011092654
[  5.0% noise]  RMSE: 100.129320 ± 1.196952  |  MSE: 10027.315862 ± 240.499206  |  NMSE: 0.0464317342 ± 0.0011136375
[ 10.0% noise]  RMSE: 100.138884 ± 1.208498  |  MSE: 10029.264583 ± 242.833724  |  NMSE: 0.0464407578 ± 0.0011244476
[ 15.0% noise]  RMSE: 100.128782 ± 1.189720  |  MSE: 10027.206391 ± 239.025248  |  NMSE: 0.0464312273 ± 0.0011068123
